In [1]:
import os
import re
import json
import time
import gc
import logging
import random
import threading
import xml.etree.ElementTree as ET

import dotenv
import pymupdf as fitz  
import requests
import chromadb

from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import hf_hub_download, HfApi, list_repo_files
from huggingface_hub.utils import disable_progress_bars

dotenv.load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")
HF_REPO_ID = os.getenv("HF_REPO_ID")

client = OpenAI(api_key=OPENAI_KEY)
hf_api = HfApi(token=HF_TOKEN)

logging.basicConfig(filename="pipeline.log", level=logging.INFO)

c:\Users\jm201\OneDrive\Documents\GitHub\Enterprise-Knowledge-Mining-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MAX_RESULTS = 1000
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
BATCH_SIZE = 100
MAX_CHUNKS_PER_PAPER = 300
CHECKPOINT_FILE = "checkpoint.json"
UPLOAD_BATCH_SIZE = 20
counter = 0


BASE_URL = "https://export.arxiv.org/api/query"

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom"
}


In [ ]:
_hf_files_cache = None
_hf_cache_lock = threading.Lock()
hf_upload_queue = []
hf_upload_lock = threading.Lock()
_arxiv_semaphore = threading.Semaphore(4)


def queue_hf_upload(local_path):
    batch = None

    with hf_upload_lock:
        hf_upload_queue.append(local_path)

        if len(hf_upload_queue) >= UPLOAD_BATCH_SIZE:
            batch = hf_upload_queue[:]
            hf_upload_queue.clear()

    if batch is not None:
        flush_batch(batch)


def flush_batch(batch):
    print(f"Uploading batch of {len(batch)} files...")

    for path in batch:
        upload_to_hf(path)
        

def get_hf_files():
    global _hf_files_cache
    with _hf_cache_lock:
        if _hf_files_cache is None:
            try:
                _hf_files_cache = set(list_repo_files(
                    repo_id=HF_REPO_ID,
                    repo_type="dataset",
                    token=HF_TOKEN
                ))
                print(f"HF repo contains {len(_hf_files_cache)} files.")
            except Exception as e:
                print(f"HF fetch failed: {e}")
                _hf_files_cache = set()
        return _hf_files_cache


def pdf_exists_in_hf(filename):
    return filename in get_hf_files()


def upload_to_hf(local_path):
    filename = os.path.basename(local_path)

    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=f"pdfs/{filename}",
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    with _hf_cache_lock:
        _hf_files_cache.add(filename)


def download_from_hf(filename, local_path):
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type="dataset",
        local_dir=os.path.dirname(local_path),
    )


def flush_hf_uploads():
    with hf_upload_lock:
        if not hf_upload_queue:
            return

        batch = hf_upload_queue[:]
        hf_upload_queue.clear()

    print(f"Uploading batch of {len(batch)} files to HF...")

    for path in batch:
        try:
            upload_to_hf(path)
        except Exception as e:
            print(f"Upload failed for {path}: {e}")

In [4]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        return set(json.load(open(CHECKPOINT_FILE)))
    return set()


def save_checkpoint(ids):
    json.dump(list(ids), open(CHECKPOINT_FILE, "w"))

In [5]:
def get_text(entry, tag):
    return entry.findtext(tag, default="", namespaces=NS).strip()


def extract_authors(entry):
    return [a.find("atom:name", NS).text.strip() for a in entry.findall("atom:author", NS)]


def extract_categories(entry):
    return [c.attrib.get("term") for c in entry.findall("atom:category", NS) if c.attrib.get("term")]


def extract_pdf_link(entry):
    return next((
        l.attrib.get("href")
        for l in entry.findall("atom:link", NS)
        if l.attrib.get("type") == "application/pdf"
    ), None)


def extract_primary_category(entry):
    primary = entry.find("arxiv:primary_category", NS)
    return primary.attrib.get("term") if primary is not None else None


def extract_paper(entry):
    return {
        "arxiv_id": get_text(entry, "atom:id").split("/")[-1],
        "title": get_text(entry, "atom:title"),
        "summary": get_text(entry, "atom:summary"),
        "published": get_text(entry, "atom:published"),
        "authors": extract_authors(entry),
        "primary_category": extract_primary_category(entry),
        "categories": extract_categories(entry),
        "pdf_link": extract_pdf_link(entry),
    }


def fetch_papers():
    url = f"{BASE_URL}?search_query=all:ai&max_results={MAX_RESULTS}"
    response = requests.get(url)
    root = ET.fromstring(response.content)
    return [extract_paper(e) for e in root.findall("atom:entry", NS)]


In [6]:
def safe_request(url):
    time.sleep(0.2)
    return requests.get(url)


def get_pdf(paper):
    filename = f"{paper['arxiv_id']}.pdf"
    local_path = f"temp/{filename}"
    os.makedirs("temp", exist_ok=True)

    if pdf_exists_in_hf(filename):
        download_from_hf(filename, local_path)
    else:
        with _arxiv_semaphore:
            r = safe_request(paper["pdf_link"])
            with open(local_path, "wb") as f:
                f.write(r.content)

        queue_hf_upload(local_path)

    return local_path

In [7]:
def parse_pdf(paper, path):
    doc = fitz.open(path)
    pages = []

    for i, page in enumerate(doc, start=1):
        pages.append({
            "text": page.get_text(),
            "raw_text": page.get_text(),
            "metadata": {
                "page_number": i,
                "page_count": len(doc),
                "format": doc.metadata.get("format", ""),
                "creationDate": doc.metadata.get("creationDate", ""),
            }
        })

    doc.close()
    return {**paper, "pages": pages}

In [8]:
def clean_text(text):
    if not text:
        return ""
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_pages(paper):
    new_pages = []
    for p in paper["pages"]:
        cleaned = clean_text(p["text"])
        if len(cleaned) > 50:
            new_pages.append({**p, "text": cleaned})
    paper["pages"] = new_pages
    return paper


In [9]:
def split_blocks(paper):
    blocks = []

    for page in paper["pages"]:
        sections = re.split(r"\n(?=[A-Z][A-Z\s]{3,})", page["text"])

        for sec in sections:
            blocks.append({
                "text": sec,
                "metadata": {
                    **page["metadata"],
                    "section": sec[:50]
                }
            })

    return {**paper, "blocks": blocks}

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)


def chunk_blocks(paper):
    chunks = []

    for b in paper["blocks"]:
        pieces = splitter.split_text(b["text"])
        for p in pieces:
            chunks.append({
                "text": p,
                "metadata": {
                    "arxiv_id": paper["arxiv_id"],
                    "title": paper["title"],
                    "page_number": b["metadata"]["page_number"],
                    "section": b["metadata"]["section"]
                }
            })

    return chunks[:MAX_CHUNKS_PER_PAPER]

In [11]:
def embed(chunks):
    texts = [c["text"] for c in chunks]
    embeddings = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        res = client.embeddings.create(model="text-embedding-3-small", input=batch)
        embeddings.extend([e.embedding for e in res.data])

    return embeddings

In [12]:
chroma = chromadb.PersistentClient(path="./db")
col = chroma.get_or_create_collection("papers")


def store(chunks, embeddings):
    ids = [f"id_{i}_{time.time()}" for i in range(len(chunks))]
    docs = [c["text"] for c in chunks]
    metas = [c["metadata"] for c in chunks]

    col.add(ids=ids, documents=docs, embeddings=embeddings, metadatas=metas)

In [13]:
def process_paper(paper, processed_ids):
    if paper["arxiv_id"] in processed_ids:
        return

    try:
        path = get_pdf(paper)
        parsed = parse_pdf(paper, path)
        cleaned = clean_pages(parsed)
        blocked = split_blocks(cleaned)
        chunks = chunk_blocks(blocked)

        if not chunks:
            return

        embeddings = embed(chunks)
        store(chunks, embeddings)

        processed_ids.add(paper["arxiv_id"])
        save_checkpoint(processed_ids)

        os.remove(path)

        del parsed, cleaned, blocked, chunks, embeddings
        gc.collect()

    except Exception as e:
        logging.error(f"FAIL {paper['arxiv_id']}: {e}")


def run():
    processed_ids = load_checkpoint()
    papers = fetch_papers()

    threads = []

    for p in papers:
        t = threading.Thread(
            target=process_paper,
            args=(p, processed_ids)
        )
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

    flush_hf_uploads()

    print("DONE")

In [ ]:
run()

HF repo contains 295 files.
Uploading batch of 20 files...


Processing Files (1 / 1): 100%|██████████| 5.13MB / 5.13MB,  713kB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Uploading batch of 20 files...


Processing Files (1 / 1): 100%|██████████|  629kB /  629kB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Uploading batch of 20 files...
Uploading batch of 20 files...


...

Processing Files (1 / 1): 100%|██████████| 5.64MB / 5.64MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Uploading batch of 20 files...


Processing Files (1 / 1): 100%|██████████|  252kB /  252kB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Uploading batch of 20 files...


Processing Files (1 / 1): 100%|██████████|  692kB /  692kB,   ???B/s  